In [60]:
# Importaciones
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn
from scipy.stats import poisson,skellam

In [61]:
# Data reading
def read_data(path):
    """To read a csv file and return a pandas df"""
    df = pd.read_csv(path)
    return df

In [62]:
from dotenv import load_dotenv
import os

load_dotenv()

sp_2526_path = os.getenv("SP_2025_2026")
sp_2627_path = os.getenv("SP_2026_2027")

In [63]:
sp1 = read_data(sp_2526_path)
sp2 = pd.read_csv(sp_2627_path)


In [64]:
# concatenar
sp_2627 = pd.concat([sp1,sp2], ignore_index=True)

sp_2627 = sp_2627[['HomeTeam','AwayTeam','FTHG','FTAG']]
sp_2627 = sp_2627.rename(columns={'FTHG': 'HomeGoals', 'FTAG': 'AwayGoals'})
#sp_2627.head()
sp_2627

,HomeTeam,AwayTeam,HomeGoals,AwayGoals
0,Girona,Vallecano,1,3
1,Villarreal,Oviedo,2,0
2,Mallorca,Barcelona,0,3
3,Alaves,Levante,2,1
4,Valencia,Sociedad,1,1
...,...,...,...,...
416,Alaves,Osasuna,5,2
417,Malaga,Levante,0,0
418,Espanol,Sevilla,1,1
419,Getafe,Celta,1,1


In [65]:
sp_2627 = sp_2627[:-10]
sp_2627.mean(numeric_only=True)

HomeGoals    1.583942
AwayGoals    1.121655
dtype: float64

In [66]:
# probability of draw between home and away team
skellam.pmf(0.0,  sp_2627.mean(numeric_only=True).iloc[0],  sp_2627.mean(numeric_only=True).iloc[1])

np.float64(0.24995302945430226)

In [67]:
# probability of home team winning by one goal
skellam.pmf(1,  sp_2627.mean(numeric_only=True).iloc[0],  sp_2627.mean(numeric_only=True).iloc[1])

np.float64(0.2322439571623345)

In [68]:
# importing the tools required for the Poisson regression model
import statsmodels.api as sm
import statsmodels.formula.api as smf

goal_model_data = pd.concat([sp_2627[['HomeTeam','AwayTeam','HomeGoals']].assign(home=1).rename(
            columns={'HomeTeam':'team', 'AwayTeam':'opponent','HomeGoals':'goals'}),
           sp_2627[['AwayTeam','HomeTeam','AwayGoals']].assign(home=0).rename(
            columns={'AwayTeam':'team', 'HomeTeam':'opponent','AwayGoals':'goals'})])

poisson_model = smf.glm(formula="goals ~ home + team + opponent", data=goal_model_data,
                        family=sm.families.Poisson()).fit()
poisson_model.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                 Generalized Linear Model Regression Results                  
==============================================================================
Dep. Variable:                  goals   No. Observations:                  822
Model:                            GLM   Df Residuals:                      776
Model Family:                 Poisson   Df Model:                           45
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -1118.6
Date:                Fri, 11 Sep 2026   Deviance:                       736.06
Time:                        16:12:41   Pearson chi2:                     636.
No. Iterations:                     5   Pseudo R-squ. (CS):             0.1816
Covariance Type:            nonrobust                                         
===========================================================================================
                              coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------
Intercept                   0.0393      0.201      0.196      0.845      -0.354       0.433
team[T.Ath Bilbao]         -0.0613      0.206     -0.298      0.766      -0.464       0.342
team[T.Ath Madrid]          0.3135      0.188      1.671      0.095      -0.054       0.681
team[T.Barcelona]           0.7512      0.173      4.346      0.000       0.412       1.090
team[T.Betis]               0.2345      0.191      1.228      0.219      -0.140       0.609
team[T.Celta]               0.0547      0.198      0.277      0.782      -0.333       0.442
team[T.Elche]               0.0568      0.200      0.283      0.777      -0.336       0.450
team[T.Espanol]            -0.0319      0.203     -0.157      0.875      -0.431       0.367
team[T.Getafe]             -0.4238      0.226     -1.872      0.061      -0.867       0.020
team[T.Girona]             -0.1581      0.215     -0.736      0.462      -0.579       0.263
team[T.La Coruna]           0.1232      0.488      0.253      0.801      -0.833       1.079
team[T.Levante]             0.0654      0.200      0.328      0.743      -0.326       0.456
team[T.Malaga]             -1.1989      1.062     -1.129      0.259      -3.279       0.882
team[T.Mallorca]            0.0311      0.204      0.152      0.879      -0.370       0.432
team[T.Osasuna]            -0.0550      0.204     -0.269      0.788      -0.456       0.346
team[T.Oviedo]             -0.5592      0.243     -2.302      0.021      -1.035      -0.083
team[T.Real Madrid]         0.5328      0.180      2.968      0.003       0.181       0.885
team[T.Santander]           0.3057      0.475      0.643      0.520      -0.626       1.237
team[T.Sevilla]             0.0593      0.199      0.297      0.766      -0.332       0.450
team[T.Sociedad]            0.2177      0.192      1.136      0.256      -0.158       0.593
team[T.Valencia]           -0.0534      0.205     -0.260      0.795      -0.455       0.349
team[T.Vallecano]          -0.0939      0.207     -0.453      0.650      -0.500       0.312
team[T.Villarreal]          0.4317      0.184      2.343      0.019       0.071       0.793
opponent[T.Ath Bilbao]      0.0607      0.183      0.331      0.740      -0.298       0.420
opponent[T.Ath Madrid]     -0.1740      0.197     -0.881      0.378      -0.561       0.213
opponent[T.Barcelona]      -0.3524      0.210     -1.680      0.093      -0.764       0.059
opponent[T.Betis]          -0.0722      0.191     -0.378      0.706      -0.447       0.303
opponent[T.Celta]          -0.1163      0.192     -0.606      0.545      -0.493       0.260
opponent[T.Elche]           0.1106      0.182      0.607      0.544      -0.247       0.468
opponent[T.Espanol]         0.0142      0.186      0.076      0.939      -0.351       0.379
opponent[T.Getafe]         -0.3316      0.204  

In [69]:
poisson_model.predict(pd.DataFrame(data={'team': 'Barcelona', 'opponent': 'Levante',
                                       'home':1},index=[1]))

1    3.57498
dtype: float64

In [70]:
poisson_model.predict(pd.DataFrame(data={'team': 'Levante', 'opponent': 'Barcelona',
                                       'home':0},index=[1]))

1    0.780555
dtype: float64

In [71]:
def simulate_match(foot_model, homeTeam, awayTeam, max_goals=10):
    home_goals_avg = foot_model.predict(pd.DataFrame(data={'team': homeTeam, 
                                                            'opponent': awayTeam,'home':1},
                                                      index=[1])).values[0]
    away_goals_avg = foot_model.predict(pd.DataFrame(data={'team': awayTeam, 
                                                            'opponent': homeTeam,'home':0},
                                                      index=[1])).values[0]
    team_pred = [[poisson.pmf(i, team_avg) for i in range(0, max_goals+1)] for team_avg in [home_goals_avg, away_goals_avg]]
    return(np.outer(np.array(team_pred[0]), np.array(team_pred[1])))
simulate_match(poisson_model, 'Barcelona', 'Levante', max_goals=3)

array([[0.01283557, 0.01001887, 0.00391014, 0.00101736],
       [0.04588692, 0.03581725, 0.01397866, 0.00363704],
       [0.08202241, 0.06402297, 0.02498671, 0.00650116],
       [0.09774284, 0.07629362, 0.02977567, 0.00774718]])

In [72]:
fcb_levante = simulate_match(poisson_model, "Barcelona", "Levante", max_goals=10)
# FCB win
np.sum(np.tril(fcb_levante, -1))

np.float64(0.8722178389759322)

In [73]:
# Draw
np.sum(np.diag(fcb_levante))

np.float64(0.08290104961335942)

In [74]:
# levante win
np.sum(np.triu(fcb_levante, 1))

np.float64(0.04367717987748622)

In [75]:
# Score
def score_probability(foot_model, home_team, away_team, home_goals, away_goals):
    match_matrix = simulate_match(
        foot_model,
        home_team,
        away_team,
        max_goals=max(home_goals, away_goals)
    )

    probability = match_matrix[home_goals][away_goals]

    return probability * 100

In [76]:
prob = score_probability(
    poisson_model,
    "Barcelona",
    "Real Madrid",
    5,
    1
)

print(prob)

1.499585440330697
